# 🔬 Séance 4 — Vision par ordinateur pour l'imagerie cellulaire

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nevermind78/IA_Biotech/blob/main/notebooks/S4_imagerie_cellpose.ipynb)

**Module : Intelligence Artificielle et Biotechnologie — Atelier 4 / 5 (2h)**

> ⚙️ **Avant de commencer :** `Exécution` → `Modifier le type d'exécution` → **GPU T4**.

## 🧬 Le problème biologique

Compter des cellules, mesurer leur taille ou repérer celles qui ont une **morphologie anormale**
(division, apoptose, noyau géant…) est une tâche quotidienne en biologie cellulaire, en criblage de
médicaments (*high-content screening*) ou en anatomopathologie. À la main, c'est **lent, fatigant et
subjectif**.

> **Question :** peut-on **automatiser** la segmentation des noyaux sur une image de microscopie en
> fluorescence, mesurer chaque cellule, détecter les cellules atypiques et produire un **rapport
> quantitatif** reproductible ?

**Image de travail :** noyaux de cellules humaines marqués à l'ADN (fluorescence), dont certaines sont
**en mitose** (image `human_mitosis` de scikit-image, ~500 × 500 pixels).

## 🎯 Objectifs de l'atelier
1. Comprendre ce qu'est une **image numérique** (une matrice de nombres)
2. Segmenter avec une méthode **classique** (seuillage d'Otsu + watershed) et en voir les limites
3. Segmenter avec **Cellpose**, un **réseau de neurones convolutif pré-entraîné**
4. **Quantifier** chaque cellule, détecter les **anomalies morphologiques**
5. Livrer un **script automatisé** qui exporte un rapport (CSV + figure)

## 🧭 Légende
▶️ exécuter · ✍️ à vous · 🔎 interpréter · 💡 solution (cellule masquée : double-cliquez)

> Les exercices de cette séance consistent surtout à **régler des paramètres** : les corrigés
> chiffrés complets figurent aussi sur la page du cours, section « Exercices et défis ».

⏱️ **Déroulé indicatif :** image numérique (15 min) · segmentation classique (25 min) · Cellpose (25 min) · quantification & anomalies (25 min) · rapport automatisé (20 min) · bonus transfer learning (10 min +)

---
## 0. Installation et imports ▶️
💡 **Lancez cette cellule dès le début** : l'installation de Cellpose prend 1 à 2 minutes.

In [ ]:
%pip install -q cellpose pooch

In [ ]:
import os, shutil, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from scipy import ndimage as ndi
from skimage import data, exposure, filters, measure, morphology, segmentation, color
from skimage.feature import peak_local_max
from sklearn.ensemble import IsolationForest

sns.set_theme(style="white")
GPU = torch.cuda.is_available()
print("✅ Environnement prêt — GPU disponible :", GPU)

---
## 1. Une image n'est qu'une matrice de nombres ▶️

In [ ]:
try:
    img = data.human_mitosis()           # noyaux humains en fluorescence (scikit-image)
except Exception as e:
    print("Source principale indisponible :", type(e).__name__, "→ téléchargement direct")
    import imageio.v3 as iio
    URL = ("https://gitlab.com/scikit-image/data/-/raw/main/"
           "AS_09125_050116030001_D03f00d0.tif")
    img = iio.imread(URL)

print("Type        :", type(img))
print("Dimensions  :", img.shape, "pixels")
print("Type des valeurs :", img.dtype, f"(de {img.min()} à {img.max()})")
print("Coin supérieur gauche (5×5 pixels) :\n", img[:5, :5])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("Image brute")
axes[1].imshow(exposure.equalize_adapthist(img), cmap="magma"); axes[1].set_title("Contraste amélioré (CLAHE)")
axes[2].hist(img.ravel(), bins=100, color="tab:purple"); axes[2].set_yscale("log")
axes[2].set_title("Histogramme des intensités"); axes[2].set_xlabel("Intensité")
for ax in axes[:2]: ax.axis("off")
plt.tight_layout(); plt.show()

### 🔎 Question 1
Sur l'histogramme, distinguez le **fond** (pixels sombres, très nombreux) et les **noyaux**. Repérez
sur l'image contrastée les cellules **en mitose** : à quoi ressemblent-elles ?

*Votre réponse :* …

---
## 2. Segmentation classique : seuillage + watershed ▶️ ✍️

1. **Lissage** gaussien pour réduire le bruit ;
2. **Seuillage d'Otsu** : trouve automatiquement l'intensité qui sépare au mieux fond et objets ;
3. **Nettoyage** : suppression des petits objets, remplissage des trous ;
4. **Watershed** (ligne de partage des eaux) sur la carte des distances pour **séparer les noyaux
   qui se touchent**.

In [ ]:
# ✍️ Paramètres de la méthode classique
SIGMA = 1.0           #@param {type:"slider", min:0, max:4, step:0.5}
TAILLE_MIN = 30       #@param {type:"slider", min:5, max:200, step:5}
DISTANCE_MIN = 5      #@param {type:"slider", min:2, max:20, step:1}

lisse = filters.gaussian(img, sigma=SIGMA, preserve_range=True)
seuil = filters.threshold_otsu(lisse)
binaire = lisse > seuil
binaire = morphology.remove_small_objects(binaire, TAILLE_MIN)
binaire = ndi.binary_fill_holes(binaire)

distance = ndi.distance_transform_edt(binaire)
pics = peak_local_max(distance, min_distance=DISTANCE_MIN, labels=measure.label(binaire))
marqueurs = np.zeros(img.shape, dtype=int)
marqueurs[tuple(pics.T)] = np.arange(1, len(pics) + 1)
labels_classique = segmentation.watershed(-distance, marqueurs, mask=binaire)

print(f"Seuil d'Otsu = {seuil:.1f}")
print(f"Objets avant watershed : {measure.label(binaire).max()}  →  noyaux après watershed : {labels_classique.max()}")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(binaire, cmap="gray"); axes[0].set_title("Masque binaire (Otsu)")
axes[1].imshow(distance, cmap="viridis"); axes[1].set_title("Carte des distances")
axes[2].imshow(color.label2rgb(labels_classique, bg_label=0)); axes[2].set_title(f"Watershed : {labels_classique.max()} noyaux")
axes[3].imshow(img, cmap="gray")
axes[3].contour(labels_classique > 0, levels=[0.5], colors="yellow", linewidths=0.6)
axes[3].set_title("Contours sur l'image")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

### 🔎 Question 2
Zoomez sur l'image (clic droit → ouvrir l'image dans un nouvel onglet). Trouvez des noyaux
**fusionnés** (sous-segmentation) ou **coupés en deux** (sur-segmentation). Que se passe-t-il si vous
mettez `DISTANCE_MIN = 2` ? `= 15` ? Pourquoi est-il difficile de trouver des paramètres qui marchent
**partout** ?

*Votre réponse :* …

In [ ]:
#@title 💡 Solution de la question 2 : effet de DISTANCE_MIN sur le comptage
def compter_classique(distance_min, sigma=SIGMA, taille_min=TAILLE_MIN):
    lisse_ = filters.gaussian(img, sigma=sigma, preserve_range=True)
    bin_ = morphology.remove_small_objects(lisse_ > filters.threshold_otsu(lisse_), taille_min)
    bin_ = ndi.binary_fill_holes(bin_)
    dist_ = ndi.distance_transform_edt(bin_)
    pics_ = peak_local_max(dist_, min_distance=distance_min, labels=measure.label(bin_))
    marq_ = np.zeros(img.shape, dtype=int)
    marq_[tuple(pics_.T)] = np.arange(1, len(pics_) + 1)
    return segmentation.watershed(-dist_, marq_, mask=bin_).max()

distances = [2, 3, 5, 8, 12, 15, 20]
comptes = [compter_classique(d) for d in distances]

plt.figure(figsize=(7, 3.5))
plt.plot(distances, comptes, marker="o", lw=2, label="Otsu + watershed")
plt.xlabel("DISTANCE_MIN (distance minimale entre deux centres)")
plt.ylabel("Nombre de noyaux détectés")
plt.title("Le comptage dépend fortement d'un paramètre réglé à la main")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

for d, n in zip(distances, comptes):
    print(f"  DISTANCE_MIN = {d:>2}  →  {n:>4} noyaux")
print()
print("→ Petite valeur : chaque bosse devient un noyau (SUR-segmentation).")
print("→ Grande valeur : les noyaux voisins fusionnent (SOUS-segmentation).")
print("→ Aucun réglage n'est bon partout : la densité varie d'une zone à l'autre de l'image.")

---
## 3. Segmentation par deep learning : Cellpose ▶️

**Cellpose** est un réseau de neurones convolutif (architecture **U-Net**) entraîné sur des milliers
d'images annotées de cellules et de noyaux, de toutes origines. Au lieu de seuiller, il prédit pour
chaque pixel la **probabilité d'appartenir à une cellule** et un **champ de flux** qui pointe vers le
centre de la cellule — ce qui sépare naturellement les cellules qui se touchent.

On l'utilise **tel quel, sans ré-entraînement** : c'est l'**apprentissage par transfert**.

In [ ]:
# ▶️ Chargement du modèle pré-entraîné
# ⏳ Au premier lancement, Cellpose télécharge ses poids (≈ 1,2 Go) : comptez 1 à 2 minutes.
from importlib.metadata import version
from cellpose import models

VERSION_CELLPOSE = int(version("cellpose").split(".")[0])
if VERSION_CELLPOSE >= 4:          # Cellpose 4 : modèle généraliste « Cellpose-SAM »
    modele_cp = models.CellposeModel(gpu=GPU)
else:                              # Cellpose 2/3 : modèle spécialisé « nuclei »
    modele_cp = models.CellposeModel(gpu=GPU, model_type="nuclei")
print("✅ Cellpose", version("cellpose"), "chargé")

def segmenter_cellpose(image, diametre=None):
    """Renvoie une image d'étiquettes : 0 = fond, 1..N = un numéro par cellule."""
    if VERSION_CELLPOSE >= 4:
        sortie = modele_cp.eval(image, diameter=diametre)
    else:
        sortie = modele_cp.eval(image, diameter=diametre, channels=[0, 0])
    return sortie[0]

In [ ]:
# ▶️ Segmentation (quelques secondes sur GPU, ~1 minute sur CPU)
t0 = time.time()
labels_cp = segmenter_cellpose(img)
print(f"Cellpose : {labels_cp.max()} noyaux détectés en {time.time() - t0:.1f} s")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, lab, titre in [(axes[0], labels_classique, "Classique (Otsu + watershed)"),
                       (axes[1], labels_cp, "Cellpose (deep learning)")]:
    ax.imshow(exposure.equalize_adapthist(img), cmap="gray")
    ax.contour(lab, levels=np.arange(0.5, lab.max() + 1), colors="yellow", linewidths=0.4)
    ax.set_title(f"{titre} : {lab.max()} noyaux"); ax.axis("off")
plt.tight_layout(); plt.show()

### 🔎 Question 3
Comparez les deux segmentations, surtout là où les noyaux **se touchent** et pour les noyaux **très
lumineux en mitose**. Laquelle est la plus proche de ce que ferait un biologiste ?
Quel est le **prix** du deep learning (temps, GPU, installation, « boîte noire ») ?

*Votre réponse :* …

In [ ]:
#@title 💡 Solution de la question 3 : comparaison chiffrée des deux méthodes
def stats(labels, nom):
    aires = measure.regionprops_table(labels, properties=["area"])["area"]
    mediane = float(np.median(aires))
    return {"méthode": nom,
            "noyaux détectés": int(labels.max()),
            "aire médiane (px)": round(mediane),
            "aire maximale (px)": int(aires.max()),
            "objets > 2× la médiane": int((aires > 2 * mediane).sum()),
            "écart-type des aires": round(float(np.std(aires)))}

lignes = [stats(labels_classique, "Otsu + watershed"), stats(labels_cp, "Cellpose")]
comparaison = pd.DataFrame(lignes).set_index("méthode")
display(comparaison)

cl, cp = lignes[0], lignes[1]
print("Lecture des chiffres obtenus :")
print(f"• Nombre de noyaux : {cl['noyaux détectés']} (classique) contre {cp['noyaux détectés']} (Cellpose).")
print(f"• Objet le plus gros : {cl['aire maximale (px)']} px contre {cp['aire maximale (px)']} px.")
print("  Un objet anormalement gros trahit deux noyaux accolés comptés comme un seul")
print("  (SOUS-segmentation) : c'est là que la méthode classique perd le plus.")
print(f"• Objets > 2× la médiane : {cl['objets > 2× la médiane']} contre {cp['objets > 2× la médiane']}.")
print()
print("⚠️ Attention à une fausse impression de propreté : `remove_small_objects(TAILLE_MIN)`")
print("   supprime d'office les petits objets de la méthode classique. Elle paraît donc plus nette,")
print("   mais ce filtre peut aussi effacer de vrais petits noyaux.")
print()
print("Prix du deep learning : installation (~1 min), 1,2 Go de poids, GPU conseillé,")
print("et un modèle « boîte noire » que l'on ne peut pas régler finement.")

---
## 4. Quantifier chaque cellule ▶️ ✍️

`regionprops_table` mesure chaque objet segmenté : surface, périmètre, excentricité (0 = cercle,
1 = segment), solidité (1 = forme convexe), intensité moyenne…

In [ ]:
# ✍️ Taille d'un pixel en micromètres (dans la vraie vie : lue dans les métadonnées du microscope)
TAILLE_PIXEL_UM = 0.5  #@param {type:"number"}

def mesurer(labels, image):
    props = measure.regionprops_table(
        labels, intensity_image=image,
        properties=["label", "area", "perimeter", "eccentricity", "solidity",
                    "intensity_mean", "centroid", "axis_major_length", "axis_minor_length"])
    t = pd.DataFrame(props)
    t["surface_um2"] = t["area"] * TAILLE_PIXEL_UM ** 2
    t["circularite"] = 4 * np.pi * t["area"] / t["perimeter"].clip(lower=1) ** 2
    return t.rename(columns={"centroid-0": "y", "centroid-1": "x"})

mesures = mesurer(labels_cp, img)
print(f"{len(mesures)} cellules mesurées")
display(mesures.head())
display(mesures[["surface_um2", "eccentricity", "solidity", "circularite", "intensity_mean"]].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(mesures["surface_um2"], bins=40, ax=axes[0], color="tab:blue"); axes[0].set_title("Surface (µm²)")
sns.histplot(mesures["intensity_mean"], bins=40, ax=axes[1], color="tab:orange"); axes[1].set_title("Intensité moyenne")
sns.scatterplot(data=mesures, x="surface_um2", y="intensity_mean", s=12, ax=axes[2])
axes[2].set_title("Intensité vs surface")
plt.tight_layout(); plt.show()

---
## 5. Détecter les morphologies anormales ▶️ ✍️

Les noyaux **en mitose** ont un ADN **condensé** : ils sont plus **petits** et beaucoup plus
**lumineux**. On utilise une **Isolation Forest** (apprentissage non supervisé) : elle isole les
cellules qui ne ressemblent pas à la majorité. Le paramètre `contamination` = proportion attendue
d'anomalies.

In [ ]:
# ✍️ Proportion attendue de cellules anormales
CONTAMINATION = 0.05  #@param {type:"slider", min:0.01, max:0.2, step:0.01}
VARIABLES = ["surface_um2", "intensity_mean", "eccentricity", "solidity"]

iso = IsolationForest(contamination=CONTAMINATION, random_state=42)
mesures["anomalie"] = iso.fit_predict(mesures[VARIABLES]) == -1
print(f"Cellules anormales : {mesures['anomalie'].sum()} / {len(mesures)}")

fig, axes = plt.subplots(1, 2, figsize=(17, 7), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].imshow(exposure.equalize_adapthist(img), cmap="gray")
masque_anormal = np.isin(labels_cp, mesures.loc[mesures["anomalie"], "label"])
axes[0].contour(labels_cp > 0, levels=[0.5], colors="deepskyblue", linewidths=0.4)
axes[0].contour(masque_anormal, levels=[0.5], colors="red", linewidths=1.2)
axes[0].set_title("Rouge : cellules anormales"); axes[0].axis("off")
sns.scatterplot(data=mesures, x="surface_um2", y="intensity_mean", hue="anomalie",
                palette={False: "deepskyblue", True: "red"}, s=18, ax=axes[1])
axes[1].set_title("Les anomalies dans l'espace des mesures")
plt.tight_layout(); plt.show()

### 🔎 Question 4
Les cellules détectées comme anormales correspondent-elles aux cellules **en mitose** que vous aviez
repérées à la question 1 ? Y a-t-il des **faux positifs** (ex. deux noyaux collés, cellule coupée au
bord de l'image) ? Comment pourrait-on les éliminer ?

*Votre réponse :* …

---
## 6. 🏁 Livrable : un script d'analyse automatisé ▶️ ✍️

On regroupe tout dans **une fonction** qui prend une image et produit le rapport. On la teste sur
**4 champs** (les 4 quarts de l'image), comme si on analysait plusieurs images d'une même lame.

In [ ]:
def analyser_image(image, nom, exclure_bords=True, dossier="rapport"):
    os.makedirs(dossier, exist_ok=True)
    labels = segmenter_cellpose(image)
    if exclure_bords:                                   # cellules coupées par le bord de l'image
        labels = segmentation.clear_border(labels)
    t = mesurer(labels, image)
    if len(t) == 0:
        return {"image": nom, "nb_cellules": 0}
    iso = IsolationForest(contamination=CONTAMINATION, random_state=42)
    t["anomalie"] = iso.fit_predict(t[VARIABLES]) == -1 if len(t) > 10 else False
    t.insert(0, "image", nom)
    t.to_csv(f"{dossier}/{nom}_cellules.csv", index=False)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(exposure.equalize_adapthist(image), cmap="gray")
    ax.contour(labels > 0, levels=[0.5], colors="deepskyblue", linewidths=0.5)
    ax.contour(np.isin(labels, t.loc[t["anomalie"], "label"]), levels=[0.5], colors="red", linewidths=1)
    ax.set_title(f"{nom} : {len(t)} cellules, {t['anomalie'].sum()} anormales"); ax.axis("off")
    fig.savefig(f"{dossier}/{nom}_segmentation.png", dpi=110, bbox_inches="tight"); plt.close(fig)

    return {"image": nom, "nb_cellules": len(t),
            "surface_moyenne_um2": t["surface_um2"].mean(), "surface_ecart_type_um2": t["surface_um2"].std(),
            "intensite_moyenne": t["intensity_mean"].mean(),
            "nb_anomalies": int(t["anomalie"].sum()), "pct_anomalies": 100 * t["anomalie"].mean()}

h, w = img.shape
champs = {"champ_1": img[:h//2, :w//2], "champ_2": img[:h//2, w//2:],
          "champ_3": img[h//2:, :w//2], "champ_4": img[h//2:, w//2:]}

rapport = pd.DataFrame([analyser_image(im, nom) for nom, im in champs.items()]).round(2)
rapport.to_csv("rapport/rapport_global.csv", index=False)
display(rapport)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, nom in zip(axes, champs):
    ax.imshow(plt.imread(f"rapport/{nom}_segmentation.png")); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ▶️ Télécharger le rapport complet (zip)
shutil.make_archive("rapport_imagerie", "zip", "rapport")
try:
    from google.colab import files
    files.download("rapport_imagerie.zip")
except ImportError:
    print("Hors Colab : rapport_imagerie.zip est dans le dossier courant")

### ✍️ Défi 1 — Votre propre image
Téléversez une image de microscopie (noyaux en fluorescence, format PNG/TIF) et analysez-la avec
`analyser_image`. Des images libres sont disponibles sur la
[Broad Bioimage Benchmark Collection](https://bbbc.broadinstitute.org) (ex. BBBC039).

In [ ]:
# ✍️ Mettez True pour téléverser votre image (dans Colab)
TELEVERSER = False  #@param {type:"boolean"}

if TELEVERSER:
    from google.colab import files
    from skimage import io as skio
    for nom_fichier in files.upload():
        mon_image = skio.imread(nom_fichier, as_gray=True)
        display(pd.DataFrame([analyser_image(mon_image, nom_fichier.split(".")[0])]).round(2))

---
## 🚀 Bonus — Transfer learning : détecter des globules rouges infectés par le paludisme

Cellpose est un réseau **déjà entraîné** que l'on utilise tel quel. Une autre forme très courante
d'apprentissage par transfert : prendre un **CNN entraîné sur ImageNet** (photos de la vie courante),
l'utiliser comme **extracteur de caractéristiques**, et n'entraîner qu'un **petit classifieur** au-dessus.

Données : images de globules rouges **infectés par *Plasmodium*** ou **sains** (NIH, ~27 000 images ;
nous en utilisons 1 000). ⏱️ ~5 minutes avec GPU (téléchargement de 340 Mo inclus).

In [ ]:
# ▶️ Bonus étape 1 : télécharger et lire un échantillon d'images
import io, random, zipfile, urllib.request
from PIL import Image

URL_PALU = "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip"
N_PAR_CLASSE = 500  #@param {type:"slider", min:100, max:2000, step:100}

if not os.path.exists("cell_images.zip"):
    print("Téléchargement (≈ 340 Mo)…")
    urllib.request.urlretrieve(URL_PALU, "cell_images.zip")

random.seed(0)
images_palu, etiquettes_palu = [], []
with zipfile.ZipFile("cell_images.zip") as z:
    for classe, valeur in [("Parasitized", 1), ("Uninfected", 0)]:
        noms = [n for n in z.namelist() if f"/{classe}/" in n and n.endswith(".png")]
        for n in random.sample(noms, N_PAR_CLASSE):
            images_palu.append(Image.open(io.BytesIO(z.read(n))).convert("RGB"))
            etiquettes_palu.append(valeur)
etiquettes_palu = np.array(etiquettes_palu)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, i in zip(axes.ravel(), list(range(8)) + list(range(N_PAR_CLASSE, N_PAR_CLASSE + 8))):
    ax.imshow(images_palu[i]); ax.axis("off")
    ax.set_title("infecté" if etiquettes_palu[i] else "sain", color="red" if etiquettes_palu[i] else "green")
plt.tight_layout(); plt.show()

In [ ]:
# ▶️ Bonus étape 2 : ResNet-18 pré-entraîné sur ImageNet comme extracteur de caractéristiques
import torchvision
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

poids = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
cnn = torchvision.models.resnet18(weights=poids)
cnn.fc = torch.nn.Identity()                 # on retire la dernière couche (les 1000 classes ImageNet)
cnn = cnn.eval().to("cuda" if GPU else "cpu")
pretraitement = poids.transforms()

@torch.no_grad()
def extraire(images, lot=64):
    feats = []
    for i in range(0, len(images), lot):
        x = torch.stack([pretraitement(im) for im in images[i:i + lot]]).to(next(cnn.parameters()).device)
        feats.append(cnn(x).cpu().numpy())
    return np.vstack(feats)

X_palu = extraire(images_palu)
print("Caractéristiques extraites :", X_palu.shape, "(1 vecteur de 512 nombres par image)")

Xtr, Xte, ytr, yte, imtr, imte = train_test_split(X_palu, etiquettes_palu, list(range(len(images_palu))),
                                                  test_size=0.25, stratify=etiquettes_palu, random_state=42)
clf_palu = LogisticRegression(max_iter=3000, C=0.5).fit(Xtr, ytr)
pred_palu = clf_palu.predict(Xte)
print(f"Exactitude sur le test : {accuracy_score(yte, pred_palu):.1%}")
ConfusionMatrixDisplay.from_predictions(yte, pred_palu, display_labels=["sain", "infecté"], cmap="Greens")
plt.show()

In [ ]:
# ▶️ Bonus étape 3 : regarder les erreurs (indispensable avant de faire confiance à un modèle !)
erreurs = [i for i, (p, t) in enumerate(zip(pred_palu, yte)) if p != t][:8]
if erreurs:
    fig, axes = plt.subplots(1, len(erreurs), figsize=(2.2 * len(erreurs), 2.8))
    for ax, i in zip(np.atleast_1d(axes), erreurs):
        ax.imshow(images_palu[imte[i]]); ax.axis("off")
        ax.set_title(f"vrai : {'inf.' if yte[i] else 'sain'}\nprédit : {'inf.' if pred_palu[i] else 'sain'}", fontsize=9)
    plt.tight_layout(); plt.show()

---
## 📝 Bilan de la séance

| Étape | Notion clé |
|---|---|
| Image = matrice | pixels, intensités, histogramme |
| Otsu + watershed | méthode classique : **paramètres réglés à la main**, fragile |
| Cellpose | **CNN U-Net pré-entraîné**, généralise à de nouvelles images |
| regionprops | de la segmentation aux **mesures quantitatives** |
| Isolation Forest | **détection d'anomalies** non supervisée |
| ResNet + régression logistique | **transfer learning** par extraction de caractéristiques |

**Livrable à rendre :** `rapport_imagerie.zip` (CSV par cellule, rapport global, figures) + réponses
aux questions 🔎 1 à 4.

➡️ **Séance suivante :** prédire la **structure 3D** d'une protéine avec AlphaFold 2 et étudier l'effet
d'une mutation.